In [6]:
#Install all required packages
#!pip install : installs Python packages from the internet
#-q: queeit mode  - reduces installation output messages
!pip install groq chromadb sentence-transformers pandas -q

print("All packages installed successfully")
print("Ready for Day 10 - Internship Finale1")

All packages installed successfully
Ready for Day 10 - Internship Finale1


In [7]:
#install required packages if not already installed (redundant with previous cell but ensures availability)
!pip install groq chromadb -q

#import all libraries
#pandas : for loading and analyzing data files (learned day 1)
import pandas as pd

#sqlite3: for running SQL queries on data (learned Day 2)
import sqlite3

#groq: for calling the groqAI API (learned on day 6)
from groq import Groq

#chromaDb: for storing and searching embeddings (learned on day 7)
import chromadb

#os: for accessing environment variables(like API keys)
import os

#json: for working with JSON formatted data
import json

print("All imports successful.")
print("Libraries loaded: pandas, sqlite3, groq, chromadb, os,json")

All imports successful.
Libraries loaded: pandas, sqlite3, groq, chromadb, os,json


In [10]:
#your api key starts with gsk
GROQ_API_KEY = "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"

#STEP 2:Create the groq client object
#Groq(): creates a connection to the Groq AI service
#api_key: the authentication creadential we just set
client = Groq(api_key=GROQ_API_KEY)

#STEP 3: define the model name
#This is the specific LLM model we use throughout the internsip
#We have used this model since Day 6 - do not change it
MODEL = "llama-3.1-8b-instant"

print("Groq client configured.")
print(f"Model: {MODEL}")
print("Status: Ready to generate AI response")

Groq client configured.
Model: llama-3.1-8b-instant
Status: Ready to generate AI response


In [11]:
#DEMONSTRATION : how a responsible prompt guardril works

#WITHOUT guardrail (responsible - AI is told to admit ignorance)
#Good Practice : always tell the AI what to do when it does not know the answer

responsible_system_prompt = """
You are a helpful data analysis assistant.
You only answer questions based on the data provided to you.
If you are not sure about something, say: I do not have information to answer that accurately.
Never make up statistics or facts that are not in the data you receive.
"""

#Test the responsible prompt with  simple question
#This test if our quardrail works prerly
test_response = client.chat.completions.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "system", "content": responsible_system_prompt},
        {"role": "user", "content": "What is the population of Mars?"}
    ]
)
ai_answer = test_response.choices[0].message.content

#Extract the AI response text
#.choices[0]: the first response option (Groq may return multiple)
#.message.content: the actual texr of the response
ai_answer = test_response.choices[0].message.content

print("=== Responsible AI test ===")
print("Question: What is the population of Mars?")
print(f"AI response (with guardrail):")
print(ai_answer)
print("\nNote: A responsible AI should admit it cannot answer this accurately.")

=== Responsible AI test ===
Question: What is the population of Mars?
AI response (with guardrail):
I do not have information to answer that accurately. The data provided does not contain information about the population of Mars.

Note: A responsible AI should admit it cannot answer this accurately.


In [12]:
#Load Datset 1: student_perfrmance.csv
#pd.read_csv(): reads a CSV file and converts it to a DataFrame
#A DataFrame is like an Excel tabke in Python
student_df = pd.read_csv("student_performance (3).csv")

#Display the first 5 rows to verify the data loaded correctly
#.head(5):show the first 5 rows of the DataFrame
print("===Student Performance Dataset===")
print(f"Shape: {student_df.shape[0]} students, {student_df.shape[1]} features")
print()
student_df.head(5)




===Student Performance Dataset===
Shape: 30 students, 11 features



,student_id,name,age,gender,branch,attendance_pct,assignment_score,midterm_score,final_score,gpa,passed
0,1,Aarav Sharma,20,Male,CSE,85,78,72,76,7.6,Yes
1,2,Priya Patel,21,Female,ECE,92,88,85,89,8.9,Yes
2,3,Rohit Kumar,20,Male,MECH,67,55,60,58,5.8,Yes
3,4,Sneha Iyer,22,Female,CSE,95,92,90,94,9.4,Yes
4,5,Vikram Singh,21,Male,CIVIL,72,62,65,63,6.3,Yes


In [14]:
#Load Dataset 2:
notes_df = pd.read_csv("college_notes.csv")

print("==== College notes Dataset ====")
print(f"Shape: {notes_df.shape[0]} notes,{notes_df.shape[1]} columns")
print()
print(notes_df[['note_id','subject','topic','difficulty']].to_string(index=False))

==== College notes Dataset ====
Shape: 15 notes,6 columns

 note_id             subject                       topic   difficulty
       1     Data Structures                      Arrays     Beginner
       2     Data Structures                Linked Lists     Beginner
       3     Data Structures                Binary Trees Intermediate
       4     Data Structures           Stacks and Queues     Beginner
       5 Database Management                  SQL Basics     Beginner
       6 Database Management               Normalization Intermediate
       7 Database Management                    Indexing Intermediate
       8    Machine Learning                  Regression Intermediate
       9    Machine Learning              Classification Intermediate
      10    Machine Learning                  Clustering     Advanced
      11  Python Programming                   Functions     Beginner
      12  Python Programming Object Oriented Programming Intermediate
      13  Python Programming   

In [15]:
#Data Quality check  - this is what we learned on Day 3
#.isnull().sum(): counts the number of missing (NaN) values in each column
print("===Data Quality Report: student_performance.csv===")
print("Missing values per column:")
print(student_df.isnull().sum())
print()

#.dtypes: shows the data type of each column
#Expected numbers should be int64 or float64, text should be object
print("Data Types:")
print(student_df.dtypes)
print()

#

===Data Quality Report: student_performance.csv===
Missing values per column:
student_id          0
name                0
age                 0
gender              0
branch              0
attendance_pct      0
assignment_score    0
midterm_score       0
final_score         0
gpa                 0
passed              0
dtype: int64

Data Types:
student_id            int64
name                 object
age                   int64
gender               object
branch               object
attendance_pct        int64
assignment_score      int64
midterm_score         int64
final_score           int64
gpa                 float64
passed               object
dtype: object



In [16]:

#This is the fast and perfect for anslysis of tasks
conn = sqlite3.connect(':memory:')

#STEP 2: load the student DataFrame into the SQL database as a table
#.to_sql():converts a Pandas DataFrame into a SQL table
#'students': the name we give to this SQL table
#conn: the database connection object
#if_exists='replace' : if the table already exists, overwrite it
#index=False: do not add an extra index column from Pandas
student_df.to_sql('students', conn, if_exists='replace', index=False)

print('SQL database created.')
print('Table "students" loaded with ',len(student_df),"rows.")

SQL database created.
Table "students" loaded with  30 rows.


In [17]:
query1 ="""
SELECT
  branch,
  COUNT(*) AS total_students,
  ROUND(AVG(gpa), 2) AS avg_gpa,
  ROUND(AVG(attendance_pct), 1) AS avg_attendance
FROM students
GROUP BY branch
ORDER BY total_students DESC;
"""

branch_analysis = pd.read_sql_query(query1, conn)
print("=== Branch wise Performance ===")
print(branch_analysis.to_string(index=False))

=== Branch wise Performance ===
branch  total_students  avg_gpa  avg_attendance
   CSE              10     7.42            80.0
   ECE               6     6.38            69.2
  MECH               5     7.22            79.4
    IT               5     8.64            89.4
 CIVIL               4     6.75            75.0


In [18]:
query2 = """
SELECT name, branch, gpa, attendance_pct, passed
FROM  students
ORDER BY gpa DESC
LIMIT 5
"""

top_students = pd.read_sql(query2, conn)
print("=== Top 5 Students by GPA ===")
print(top_students.to_string(index=False))

=== Top 5 Students by GPA ===
            name branch  gpa  attendance_pct passed
  Meera Krishnan     IT  9.5              96    Yes
      Sneha Iyer    CSE  9.4              95    Yes
Lakshmi Chandran    CSE  9.2              94    Yes
    Swathi Menon     IT  9.1              93    Yes
     Priya Patel    ECE  8.9              92    Yes


In [19]:
query3 ="""
SELECT
   passed,
   COUNT(*) AS student_count,
   ROUND(AVG(gpa), 2) AS avg_gpa
FROM students
GROUP BY passed
"""

pass_fail = pd.read_sql(query3, conn)
print("=== Passing vs Fail statistics ===")
print(pass_fail.to_string(index=False))

total = len(student_df)
passed = len(student_df[student_df['passed'] == 'Yes']) # Corrected 'yes' to 'Yes'
pass_rate = round(passed / total * 100, 1)

print(f"Overall Pass Rate : {pass_rate}% ({passed}/{total} students)")

=== Passing vs Fail statistics ===
passed  student_count  avg_gpa
    No              3     4.47
   Yes             27     7.61
Overall Pass Rate : 90.0% (27/30 students)


In [20]:
branch_summary = ""
for _, row in branch_analysis.iterrows():
  #iterrows(): loops through each row of a DataFrame
  #row['column_name']:accessess a specific value in that row
  branch_summary += f" - {row['branch']}: {row['total_students']} students, Avg GPA: {row['avg_gpa']}, Avg Attendance: {row['avg_attendance']}%\n"

#Build top students summary
top_summary = ""
for _, row in top_students.iterrows():
  top_summary += f" - {row['name']} ({row['branch']}): GPA {row['gpa']}%\n"

#assemble the complete data summary
data_summary = f"""
STUDENT PERFORMANCE DATA SUMMARY
Total Students: {total}
Overall Pass Rate: {pass_rate}%
Average GPA actoss all students: {round(student_df['gpa'].mean(), 2)}

Performance by Branch:
{branch_summary}
Top 5 Students by GPA:
{top_summary}
"""

print("=== Data Summary ===")
print(data_summary)


=== Data Summary ===

STUDENT PERFORMANCE DATA SUMMARY
Total Students: 30
Overall Pass Rate: 90.0%
Average GPA actoss all students: 7.29

Performance by Branch:
 - CSE: 10 students, Avg GPA: 7.42, Avg Attendance: 80.0%
 - ECE: 6 students, Avg GPA: 6.38, Avg Attendance: 69.2%
 - MECH: 5 students, Avg GPA: 7.22, Avg Attendance: 79.4%
 - IT: 5 students, Avg GPA: 8.64, Avg Attendance: 89.4%
 - CIVIL: 4 students, Avg GPA: 6.75, Avg Attendance: 75.0%

Top 5 Students by GPA:
 - Meera Krishnan (IT): GPA 9.5%
 - Sneha Iyer (CSE): GPA 9.4%
 - Lakshmi Chandran (CSE): GPA 9.2%
 - Swathi Menon (IT): GPA 9.1%
 - Priya Patel (ECE): GPA 8.9%




In [21]:
system_prompt = """
You are an expert academic data analyst woeking for an engineering college.
You receive student performance summaries and provide clear, actinable insights.
Always base your analysis strictly on the data provided.
If you are uncertain about something, say so clearly.
Format your response with numbered points for clarity.
"""

#User message : the actua; question we ask the AI
user_message = f"""
Here is the student performance data for this semester:

{data_summary}

Please provide:
1. Three key insights from this data
2. Which branch needs the most improvement?
3. One recommendation for the college principal
"""

#Make the API call to Groq
#model: the LLM we want to use
# max_tokens: maximum length of the AI response (1 token = 0.75 words)
#messages: the conversation history - we send system role + user message
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=600,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
)

#Extract the AI-generated analysis
ai_analysis = response.choices[0].message.content

print("=" * 60)
print("AI - GENERATED DATA ANALYSIS")
print("=" * 60)
print(ai_analysis)

AI - GENERATED DATA ANALYSIS
Based on the student performance data provided, here are my analysis and insights:

**1. Three key insights from this data:**

1. **The IT branch is the most outstanding with exceptional student performance**: The IT branch has the highest average GPA (8.64) and attendance rate (89.4%), indicating that the students in this branch are consistently performing exceptionally well in terms of GPA and attending classes regularly.

2. **Attendance is crucial for student academic success**: Although the overall pass rate is satisfactory (90.0%), there is a noticeable trend of lower attendance rates among the ECE and CIVIL branches (69.2% and 75.0%, respectively). This suggests that regular attendance is vital for student academic performance.

3. **The ECE branch requires immediate attention**: The ECE branch has the lowest average GPA (6.38) among the five branches, indicating that these students may need additional support to improve their academic performance.



In [22]:
system_prompt = """
You are an expert academic data analyst woeking for an engineering college.
You receive student performance summaries and provide clear, actinable insights.
Always base your analysis strictly on the data provided.
If you are uncertain about something, say so clearly.
Format your response with numbered points for clarity.
"""

#User message : the actua; question we ask the AI
user_message = f"""
Here is the student performance data for this semester:

{data_summary}

Please provide:
1. Three key insights from this data
2. Which branch needs the most improvement?
3. One recommendation for the college principal
"""

#Make the API call to Groq
#model: the LLM we want to use
# max_tokens: maximum length of the AI response (1 token = 0.75 words)
#messages: the conversation history - we send system role + user message
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=600,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
)

#Extract the AI-generated analysis
ai_analysis = response.choices[0].message.content

print("=" * 60)
print("AI - GENERATED DATA ANALYSIS")
print("=" * 60)
print(ai_analysis)

AI - GENERATED DATA ANALYSIS
Based on the student performance data provided, here are three key insights:

1. **IT students are outperforming other branches in terms of GPA**: The average GPA of IT students (8.64) is significantly higher than the overall average GPA across all branches (7.29). This suggests that the IT program may be better positioned to provide a high-quality educational experience.

2. **CSE and IT students have relatively better attendance**: Students from CSE and IT (average attendance of 80.0% and 89.4%, respectively) have better attendance rates compared to other branches. This could indicate that students in these programs are more motivated and engaged in the learning process.

3. **There is a large gap in average GPAs between IT and ECE students**: The average GPA difference between IT (8.64) and ECE (6.38) is 2.26. This significant disparity may indicate a need for targeted interventions and support to improve the performance of ECE students.

Regarding which

In [23]:
try:
    validation_response = client.chat.completions.create(
        model=MODEL,
        max_tokens=10,
        messages=[
            {"role": "user", "content": "Hello"}
        ]
    )
    print("Groq API Key is VALID! Successfully made an API call.")
except Exception as e:
    print(f"Groq API Key is INVALID. Encountered an error: {e}")
    print("Please re-check your GROQ_API_KEY in cell 'h9twxC28Gt8L' and ensure it's correct.")

Groq API Key is VALID! Successfully made an API call.
